# Eurex GraphQL API Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eurex-api/api-explorer/blob/main/notebooks/eurex_graphql_tutorial.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/eurex-api/api-explorer/main?filepath=notebooks/eurex_graphql_tutorial.ipynb)

This notebook provides a comprehensive guide to interacting with the Eurex GraphQL API using Python. We will cover authentication, executing queries, handling pagination/looping, and analyzing the results with `pandas`.

## 1. Setup and Authentication

First, we need to install the necessary libraries and set up our API credentials.

In [ ]:
!pip install requests pandas

In [ ]:
import requests
import pandas as pd
import json

# Replace with your actual API key
API_KEY = 'YOUR_API_KEY_HERE'
ENDPOINT = 'https://api.developer.deutsche-boerse.com/eurex-prod-graphql/'

headers = {
    'Content-Type': 'application/json',
    'X-DBP-APIKEY': API_KEY
}

## 2. Basic Query

Let's start with a simple query to fetch some contracts for a specific product (e.g., 'FDAX').

In [ ]:
query = """
query {
  Contracts(filter: { Product: { eq: "FDAX" } }) {
    date
    data {
      ISIN
      Contract
      ExpirationDate
    }
  }
}
"""

response = requests.post(ENDPOINT, json={'query': query}, headers=headers)

if response.status_code == 200:
    result = response.json()
    # Accessing the data
    contracts = result['data']['Contracts'][0]['data']
    df = pd.DataFrame(contracts)
    print(f"Fetched {len(df)} contracts for FDAX")
    display(df.head())
else:
    print(f"Error: {response.status_code}")
    print(response.text)

## 3. Fetching All Contracts (Looping over Products)

To fetch all available contracts across multiple products, it's recommended to first get a list of products and then iterate through them. This approach respects API limits and ensures you can retrieve the full dataset systematically.

In [ ]:
def get_all_products():
    query = """
    query {
      Products {
        data {
          Product
        }
      }
    }
    """
    response = requests.post(ENDPOINT, json={'query': query}, headers=headers)
    if response.status_code == 200:
        data = response.json()['data']['Products'][0]['data']
        return [p['Product'] for p in data]
    return []

def fetch_contracts_for_product(product_code):
    query = """
    query($product: String!) {
      Contracts(filter: { Product: { eq: $product } }) {
        data {
          Product
          ISIN
          Contract
          ExpirationDate
          AssetID
        }
      }
    }
    """
    variables = {'product': product_code}
    response = requests.post(ENDPOINT, json={'query': query, 'variables': variables}, headers=headers)
    if response.status_code == 200:
        result = response.json()
        if 'data' in result and result['data']['Contracts']:
            return result['data']['Contracts'][0]['data']
    return []

# Example: Fetch for a few products to demonstrate
sample_products = ['FDAX', 'FGBL', 'FESX']
all_contracts = []

for prod in sample_products:
    print(f"Fetching contracts for {prod}...")
    contracts = fetch_contracts_for_product(prod)
    all_contracts.extend(contracts)

master_df = pd.DataFrame(all_contracts)
print(f"\nTotal contracts fetched: {len(master_df)}")
display(master_df.sample(min(10, len(master_df))))

## 4. Data Analysis

Once we have the data in a pandas DataFrame, we can perform various analyses. For example, counting contracts per product or sorting by expiration date.

In [ ]:
if not master_df.empty:
    # Convert ExpirationDate to datetime
    master_df['ExpirationDate'] = pd.to_datetime(master_df['ExpirationDate'])
    
    # Group by Product
    stats = master_df.groupby('Product').size().reset_index(name='Contract Count')
    print("Contracts per Product:")
    display(stats)
    
    # Find the next expiring contract
    next_expiring = master_df.sort_values('ExpirationDate').iloc[0]
    print(f"\nNext contract to expire: {next_expiring['Contract']} ({next_expiring['Product']}) on {next_expiring['ExpirationDate'].date()}")

## Conclusion

This notebook demonstrated the basics of using the Eurex GraphQL API. You can extend these examples by exploring the full schema in the [API Explorer](https://eurex-api.github.io/api-explorer/).